In [1]:
# list the path to the results database for all sessions here
csv_paths = ['Z:\\Jasmine_Laurence\\Experimental_Data\\JAL004\\004_flip_2023_09_03T12_04_16\\processed_data\\models\\replay\\replay_SS_decoder\\replay_results.csv',
 'Z:\\Jasmine_Laurence\\Experimental_Data\\JAL004\\004_flipppuf19sept_2023_09_19T14_10_56\\processed_data\\models\\replay\\replay_SS_decoder\\replay_results.csv',
 'Z:\\Jasmine_Laurence\\Experimental_Data\\JAL004\\JAL004_flip_rotated_2023_08_28T09_36_04\\processed_data\\models\\replay\\replay_SS_decoder\\replay_results.csv',
 'Z:\\Jasmine_Laurence\\Experimental_Data\\JAL004\\004_flip_puff2_2023_09_11T09_32_25\\processed_data\\models\\replay\\replay_SS_decoder\\replay_results.csv',
 'Z:\\Jasmine_Laurence\\Experimental_Data\\JAL005\\005_flip1_2023_09_08T07_36_54\\processed_data\\models\\replay\\replay_SS_decoder\\replay_results.csv',
 'Z:\\Jasmine_Laurence\\Experimental_Data\\JAL005\\005_flippuff3_2023_09_21T11_11_13\\processed_data\\models\\replay\\replay_SS_decoder\\replay_results.csv',
 'Z:\\Jasmine_Laurence\\Experimental_Data\\JAL006\\JAL006_shelter_barrier_flip_6_2024_03_28T10_54_20\\processed_data\\models\\replay\\replay_SS_decoder\\replay_results.csv',
 'Z:\\Jasmine_Laurence\\Experimental_Data\\JAL006\\JAL006_shelter_barrier_flip_3_2024_03_21T11_20_34\\processed_data\\models\\replay\\replay_SS_decoder\\replay_results.csv',
 'Z:\\Jasmine_Laurence\\Experimental_Data\\JAL006\\JAL006_barrier_flip2_2024_03_18T11_53_29\\processed_data\\models\\replay\\replay_SS_decoder\\replay_results.csv',
 'Z:\\Jasmine_Laurence\\Experimental_Data\\JAL006\\JAL006_shelter_barrier_flip_5_2024_03_25T11_05_33\\processed_data\\models\\replay\\replay_SS_decoder\\replay_results.csv',
 'Z:\\Jasmine_Laurence\\Experimental_Data\\JAL007\\JAL007_shelter_barrier_flip_8_2024_04_09T10_07_45\\processed_data\\models\\replay\\replay_SS_decoder\\replay_results.csv',
 'Z:\\Jasmine_Laurence\\Experimental_Data\\JAL007\\JAL007_shelter_barrier_flip_5_2024_03_22T11_15_43\\processed_data\\models\\replay\\replay_SS_decoder\\replay_results.csv',
 'Z:\\Jasmine_Laurence\\Experimental_Data\\JAL007\\JAL007_barrierflip2_2024_03_12T11_18_26\\processed_data\\models\\replay\\replay_SS_decoder\\replay_results.csv',
 'Z:\\Jasmine_Laurence\\Experimental_Data\\JAL007\\JAL007_shelter_barrier_flip_9_2024_04_16T11_13_05\\processed_data\\models\\replay\\replay_SS_decoder\\replay_results.csv',
 'Z:\\Jasmine_Laurence\\Experimental_Data\\JAL007\\JAL007_shelter_barrier_flip_100_2024_04_23T09_59_40\\processed_data\\models\\replay\\replay_SS_decoder\\replay_results.csv',
 'Z:\\Jasmine_Laurence\\Experimental_Data\\JAL008\\JAL008_shelter_barrier_flip_1_2024_04_25T11_27_42\\processed_data\\models\\replay\\replay_SS_decoder\\replay_results.csv',
 'Z:\\Jasmine_Laurence\\Experimental_Data\\JAL008\\JAL008_shelter_barrier_flip_2_2024_04_29T12_14_54\\processed_data\\models\\replay\\replay_SS_decoder\\replay_results.csv',
 'Z:\\Jasmine_Laurence\\Experimental_Data\\JAL008\\JAL008_shelter_barrier_flip_3_2024_05_07T10_16_26\\processed_data\\models\\replay\\replay_SS_decoder\\replay_results.csv',
 'Z:\\Jasmine_Laurence\\Experimental_Data\\JAL008\\JAL008_shelter_barrier_flip_5_2024_05_14T10_18_03\\processed_data\\models\\replay\\replay_SS_decoder\\replay_results.csv',
 'Z:\\Jasmine_Laurence\\Experimental_Data\\JAL008\\JAL008_shelter_barrier_flip_4_2024_05_10T11_47_47\\processed_data\\models\\replay\\replay_SS_decoder\\replay_results.csv']

session_names = ["JAL4_3rdSept","JAL4_19thSept","JAL4_28aug","JAL4_11thSept",
    "JAL5_8thSept","JAL5_21stSept",
    "JAL6_28mar", "JAL6_flip4_21mar", "JAL6_flip3_18mar", "JAL6_flip5_25mar", 
    "JAL7_sesh8_9apr", "JAL7_flip5_22mar", "JAL7_flip2_12mar", "JAL7_sesh9_16apr", "JAL7_23apr",
    "JAL8_flip1_25apr", "JAL8_flip2_29apr", "JAL8_flip3_7may",  "JAL8_14may", "JAL8_flip4_10may"] 

print(len(csv_paths), len(session_names))

20 20


In [ ]:
"""Imports"""
%load_ext autoreload
%autoreload 2
import os
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
np.warnings = warnings
from replay_trajectory_classification import SortedSpikesDecoder, Environment, RandomWalk
import matplotlib
import matplotlib.gridspec as gridspec
matplotlib.rcParams['font.family'] = 'Arial'
matplotlib.rcParams['font.size'] = 15
matplotlib.rcParams['pdf.fonttype'] = 42  # Ensure fonts are embedded in PDF

import sys
sys.path.append(r"c:\Users\Jasmine\Documents\GitHub\JAL2_subgoal_pipeline")
from behave_analysis.analyze.Replay.SSdecoder_utils import predicted_position_from_posterior, compute_rmse
from behave_analysis.analyze.Replay.SSdecode_plotting import plot_predicted_vs_actual, plot_mouse_behaviour, plot_behavioral_vars
from behave_analysis.analyze.results_database_utils import check_database_for_same_run
from settings.settings_analyze_efizz import Settings_ae
from settings.settings_overrides import settings_overrides

# make a list of the settings and what they shoul be for the experiment we're interested in
true_settings = {'replay_cells': "all",
            'replay_train_condition': 'shelter_only',
            'replay_test_condition': 'shelter_only',
            'replay_cells': "all",  # 'all','hdir','escape_tuned'
            'replay_template_variable': "escape",  # to make the order template of the replay sequence
            'replay_decoder_variable': "escape",  # 'shelter_dist' or 'escape' or 'speed' or '2D_position
            'replay_decoder_train_time_period': "correct_full_homing&escape",  # 'homing&escape', "correct_<>", "error_<>", "full_<>"
            'replay_decoder_test_time_period': "homing&escape",  #  'error_homing&escape', 'before_homing','in_shelter_after_escape','outside_shelter','stationary_outside_shelter','in_shelter'
            'replay_template_match_method': "SS_decoder",
            'stim_type': 'audio',
            'cluster_type': 'good',
            'condition_types': 'experimental_conditions',
            'compartment_split': ['all']
            }
tuning_settings = {'ep_bins': 25,
            'ep_no_stationary': False,
            'ep_interpolation_mult': 2,
            'ep_gaussian_fitting': False,
            'ep_compute_loo_reliability': False,
            'ep_tuned_compare_method': 'euclidean',
            'ep_tuned_stats': 'bootstrap',
            'ep_tuned_stats_samples': 100,
            'linshift_min_step': 120,
            'linshift_step': 80,
            'linshift_step_n': 100,
            'stim_type': 'audio',
            'cluster_type': 'good',
            'condition_types': 'experimental_conditions',
            'compartment_split': ['all']}
Settings_ae = settings_overrides(Settings_ae, {'redo_compute': False})
conditions = ['shelter_only', 'barrier_pre_flip', 'barrier_post_flip']
LM_path = r"C:\Users\Jasmine\Dropbox\BrancoLab\presentation\LM_18mar2026"
types = ['all', 'full_route', 'first_leg', 'second_leg']
stat = 'zscore_peak'
posterior = 'causal'

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
def run_decoder(data, save_file):    
    """This function runs the decoder on the provided data and settings, and saves the results to the specified file.
    INPUTS:
        data: dictionary saved by running analyze_efizz Replay SSdecoder
        save_file; path to the file that the results will be saved to. Usually the same path as data, but ending in _results.npz"""
    # set up parameters for decoder
    test_position_onsets = np.concatenate((np.array([0]), np.where(np.diff(data['test_segments']) == 1)[0]+1))
    test_position_offsets = np.concatenate((test_position_onsets[1:], [len(data['test_segments'])]))
    train_position_onsets = np.concatenate((np.array([0]), np.where(np.diff(data['train_segments']) == 1)[0]+1))
    train_position_offsets = np.concatenate((train_position_onsets[1:], [len(data['train_segments'])]))

    # set up environment parameters
    nbins = np.unique(data['train_position'])
    linear_track_env = Environment(place_bin_size=1,  # For binned data use 1, Adjust based on your actual bin size
                                    # track_graph=my_linear_graph,  # Define a linear graph with 25 nodes
                                    edge_order=[(i, i + 1) for i in range(int(max(nbins)))],  # Sequential edges
                                    edge_spacing=None,  # No gaps between edges
                                    position_range=[(0, len(nbins))],  # 0 to 100% of escape route
                                    infer_track_interior=False,  # Data is already binned
                                    fill_holes=False,
                                    dilate=False,
                                    bin_count_threshold=1)

    transition_type = RandomWalk(movement_var=1) # smooth continuous movement, 1-2 bins varriance between steps in the behaviour

    decoder = SortedSpikesDecoder(environment=linear_track_env,
                                    transition_type=transition_type,
                                    sorted_spikes_algorithm='spiking_likelihood_kde', # how the tuning curves are estimated
                                    sorted_spikes_algorithm_params={'block_size': None,
                                                                    'position_std': 1.0, # run a test to see if this should be a float or a list
                                                                    'use_diffusion': False})

    """Train and decode"""
    decoder.fit(data['train_position'], data['train_spikes'])

    # predict behaviour in test, but do it in segments to avoid weirdness a t the discontinuities between trials
    test_time = np.arange(0, data['test_time'].shape[0] / (1 / 0.001), 0.001)
    for idx, (start, end) in enumerate(zip(test_position_onsets, test_position_offsets)):
        # print(f"Predicting test data for segment {idx+1}/{len(test_position_onsets)}...")
        test_results = decoder.predict(data['test_spikes'][start:end], time=test_time[start:end], use_gpu=True)
        if idx == 0:
            test_res = np.array(test_results.causal_posterior)
        else:
            test_res = np.concatenate((test_res, np.array(test_results.causal_posterior)), axis=0)
    # extract predicted position from posterior, default uses "weighted average"
    test_predicted = predicted_position_from_posterior(test_res)

    # predict behaviour in test, but do it in segments to avoid weirdness a t the discontinuities between trials
    train_time = np.arange(0, data['train_time'].shape[0] / (1 / 0.001), 0.001)
    for idx, (start, end) in enumerate(zip(train_position_onsets, train_position_offsets)):
        # print(f"Predicting training data for segment {idx+1}/{len(train_position_onsets)}...")
        train_results = decoder.predict(data['train_spikes'][start:end], time=train_time[start:end], use_gpu=True)
        if idx == 0:
            train_res = np.array(train_results.causal_posterior)
        else:
            train_res = np.concatenate((train_res, np.array(train_results.causal_posterior)), axis=0)
    # extract predicted position from posterior, default uses "weighted average"
    train_predicted = predicted_position_from_posterior(train_res)

    """Compute RMSE for train and test data predictions"""
    # test data
    test_rmse = np.zeros(len(test_position_onsets))
    for idx in range(len(test_position_onsets)):
        test_rmse[idx] = compute_rmse(data['test_position'][test_position_onsets[idx]:test_position_offsets[idx]], 
                        test_predicted[test_position_onsets[idx]:test_position_offsets[idx]])

    # train_data
    train_rmse = np.zeros(len(train_position_onsets))
    for idx in range(len(train_position_onsets)):
        train_rmse[idx] = compute_rmse(data['train_position'][train_position_onsets[idx]:train_position_offsets[idx]], 
                        train_predicted[train_position_onsets[idx]:train_position_offsets[idx]])

    # save results
    np.savez(save_file,
            test_causal_posterior=test_res,
            test_acausal_posterior=test_res,
            train_causal_posterior=train_res,
            train_acausal_posterior=train_res,
            train_rmse=train_rmse,
            test_rmse=test_rmse,
            train_predicted=train_predicted,
            test_predicted=test_predicted,
            test_position_offsets=test_position_offsets, # where trials or segments start and end in resampled neural data time
            test_position_onsets=test_position_onsets,
            train_position_onsets=train_position_onsets,
            train_position_offsets=train_position_offsets,
            test_frame_onsets = np.where(np.diff(data['test_mask'].astype(int))>0)[0] + 1, # where trials or segments start and end in original behaviour time
            test_frame_offsets = np.where(np.diff(data['test_mask'].astype(int))<0)[0] + 1,
            train_frame_onsets = np.where(np.diff(data['train_mask'].astype(int))>0)[0] + 1,
            train_frame_offsets = np.where(np.diff(data['train_mask'].astype(int))<0)[0] + 1)

def plot_neural_data_homing(ax, dataset, h_indices, all_data, tuning, settings, return_durations=False):
    """
    INPUTS:
        h_indices: list of homing indices (within this condition) to concatenate
        dataset: 'train' or 'test'
        tuning: list of the two datasets that were used for tuning (e.g. true_settings['replay_decoder_variable'] + ' in ' + true_settings['replay_decoder_train_time_period'])
        all_data: dictionary of output of escapepatterntuning for the two tuning vars in tuning"""
    if isinstance(h_indices, int):
        h_indices = [h_indices]  # backward compatible
    
    c_test = conditions.index(str(settings[f'replay_{dataset}_condition']))
    c_train = conditions.index(str(settings[f'replay_train_condition']))
    sig_cells = all_data[tuning[0]]['sig_escape'][c_train, :]
    key = true_settings['replay_decoder_variable'] + ' in ' + true_settings[f'replay_decoder_{dataset}_time_period']

    # homing onsets in neural matrix time
    onsets = np.where(np.diff(all_data[key]['results']['homing_vector'].astype(int)) > 0)[0] + 1
    offsets = np.where(np.diff(all_data[key]['results']['homing_vector'].astype(int)) < 0)[0] + 1
    escape = all_data[key]['results']['escape_vector'][onsets]
    durations = offsets - onsets
    first = np.array([0])
    for d in durations[:-1]:
        first = np.append(first, d + first[-1])
    condition_mask = (all_data[key]['results']['condition'][first]).astype(int) == c_test
    onset_c = first[condition_mask]
    durations_c = durations[condition_mask]
    escape_c = escape[condition_mask]

    # mean/std from combined tuning neural matrices for z-scoring
    mean_fr = np.nanmean(np.append(
        all_data[tuning[0]]['results']['neural_matrix'][:, all_data[tuning[0]]['results']['condition'] == c_train],
        all_data[tuning[1]]['results']['neural_matrix'][:, all_data[tuning[1]]['results']['condition'] == c_test],
        axis=1), axis=1)
    std_fr = np.nanstd(np.append(
        all_data[tuning[0]]['results']['neural_matrix'][:, all_data[tuning[0]]['results']['condition'] == c_train],
        all_data[tuning[1]]['results']['neural_matrix'][:, all_data[tuning[1]]['results']['condition'] == c_test],
        axis=1), axis=1)

    # sort order from tuning curve peak
    tuning_hm = all_data[tuning[0]]['results']['fr_full'][c_train, sig_cells, :]
    tuning_hm = np.divide(tuning_hm - mean_fr[sig_cells, np.newaxis],
                          std_fr[sig_cells, np.newaxis],
                          out=np.zeros_like(tuning_hm, dtype=np.float64),
                          where=std_fr[sig_cells, np.newaxis] != 0)
    sort_idx = np.argsort(np.argmax(tuning_hm, axis=1))

    # concatenate neural data for selected homings
    heatmap_parts = []
    boundary_positions = []  # for axvlines between homings
    cumulative_width = 0
    is_escape = []
    for h_idx in h_indices:
        part = all_data[key]['results']['neural_matrix'][:, onset_c[h_idx]:onset_c[h_idx]+durations_c[h_idx]]
        part = part[sig_cells, :]
        part = np.divide(part - mean_fr[sig_cells, np.newaxis],
                         std_fr[sig_cells, np.newaxis],
                         out=np.zeros_like(part, dtype=np.float64),
                         where=std_fr[sig_cells, np.newaxis] != 0)
        if cumulative_width > 0:
            boundary_positions.append(cumulative_width)
        cumulative_width += part.shape[1]
        heatmap_parts.append(part)
        is_escape.append(escape_c[h_idx])

    heatmap = np.concatenate(heatmap_parts, axis=1)
    ax.imshow(heatmap[sort_idx, :], cmap="gray_r", aspect="auto", interpolation="none",
              vmin=-.5, vmax=1, extent=(0, heatmap.shape[1]/80, heatmap.shape[0], 0))
    for b in boundary_positions:
        ax.axvline(x=b/80, color='red', linestyle='--', linewidth=2)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Neurons')

    if len(is_escape) == 1:
        if is_escape[0]:
            ax.set_title(f"Escape!")

    if return_durations:
        return is_escape, [p.shape[1] for p in heatmap_parts]
    else:
        return is_escape

def homie_characteristics(vdf, onsets, offsets):
    start_loc = np.empty((len(onsets),2))
    end_loc = np.empty((len(onsets),2))
    path_length = np.empty(len(onsets))
    for i, (o, off) in enumerate(zip(onsets, offsets)):
        start_loc[i] = [vdf['mouse_x_position'].to_numpy()[o], vdf['mouse_y_position'].to_numpy()[o]]
        end_loc[i] = [vdf['mouse_x_position'].to_numpy()[off], vdf['mouse_y_position'].to_numpy()[off]]
        path_length[i] = np.sum(np.sqrt(np.diff(vdf['mouse_x_position'].to_numpy()[o:off])**2 + np.diff(vdf['mouse_y_position'].to_numpy()[o:off])**2))
    return start_loc, end_loc, path_length

def compute_tuning_stat(stat: str, shifted_matrix: np.array, shift0: int, neural_matrix=None, condition=None):
    """
    INPUTS:
        shifted_matrix: a matrix of (shifts,conditions,n_neurons,n_bins) includes the zero shift!
            most commonly this will be data['y_fitted_shift'], but could also be data['fr_shift']
        shift0: which of the shifts of shifted_matrix is the zero shift
        stat: a string defining which statistic we want to compute
                'zscore_peak' - the peak of the zscored trace
                'peak' - the peak of the trace (can find a peak even for very flat curves)
                'peak_to_mean' - the ratio of the peak to the mean firing of the tuning curve (high values for very low firing cells!)
        if stat == 'zscore_peak' need to pass:
            neural_matrix: the original neural matrix used to compute shifted_matrix (neurons x time)
            condition: vector of length time of the condition at each time point used to compute shifted_matrix
    """
    if stat == "peak_to_mean":
        peak = np.nanmax(shifted_matrix, axis=3)
        mean = np.nanmean(shifted_matrix, axis=3)
        shift_stat = np.divide(peak, mean, out=np.zeros_like(peak, dtype=np.float64), where=mean != 0)
    elif stat == "peak":
        shift_stat = np.nanmax(shifted_matrix, axis=3)
    elif stat == "zscore_peak":
        assert (neural_matrix is not None) & (condition is not None), "Need to pass neural_matrix and condition to compute zscore_peak"
        mean_fr = np.zeros((shifted_matrix.shape[1], shifted_matrix.shape[2]))  # condition x neuron
        std_fr = np.zeros((shifted_matrix.shape[1], shifted_matrix.shape[2]))  # condition x neuron
        for c in np.unique(condition):
            mean_fr[int(c), :] = np.nanmean(neural_matrix[:, condition == int(c)], axis=1)
            std_fr[int(c), :] = np.nanstd(neural_matrix[:, condition == int(c)], axis=1)
        # transform shifted_matrix to z-scores using mean and std of original neural matrix, extended to all shifts and bins
        zscored = np.divide(
            shifted_matrix - mean_fr[np.newaxis, :, :, np.newaxis],
            std_fr[np.newaxis, :, :, np.newaxis],
            out=np.zeros_like(shifted_matrix, dtype=np.float64),
            where=std_fr[np.newaxis, :, :, np.newaxis] != 0,
        )
        shift_stat = np.nanmax(zscored, axis=3)
    real_stat = shift_stat[shift0, :, :]
    shift_stat = np.delete(shift_stat, shift0, axis=0)

    return real_stat, shift_stat

def choose_homie_types(start_loc, end_loc, type, barrier_location = []):
    if type == 'all':
        return np.arange(len(start_loc))
    elif type == 'full_route':
        return np.where(np.logical_and((start_loc[:,1] < 300), (end_loc[:,1] > 800)))[0]
    elif type == 'first_leg':
        start = start_loc[:,1] < 300 # threat zone
        end = end_loc[:,1] < 510 # north of barrier
        dist_to_barrier = np.sqrt((end_loc[:,0] - barrier_location[0])**2 + (end_loc[:,1] - barrier_location[1])**2) < 50
        return np.where(np.logical_and(start, np.logical_or(end, dist_to_barrier)))[0]
    elif type == 'second_leg':
        start = start_loc[:,1] > 514 # south of barrier
        end = end_loc[:,1] > 800 # target zone
        dist_to_barrier = np.sqrt((start_loc[:,0] - barrier_location[0])**2 + (start_loc[:,1] - barrier_location[1])**2) < 50
        return np.where(np.logical_and(np.logical_or(start, dist_to_barrier), end))[0]
    
"""Plot groups of homing by TYPE"""

def homie_overview_plot(h_type, dataset, vdf, results, data, all_data, tuning, settings, LM_path, condition, session_name, posterior):
    sampling_frequency = 1 / settings['replay_state_space_decoder_bin_size'] # in Hz, bin_width is in ms
    start, end, _ = homie_characteristics(vdf, results[f'{dataset}_frame_onsets'], results[f'{dataset}_frame_offsets'])

    homies = choose_homie_types(start, end, type = h_type, barrier_location = settings[f'barrier_{dataset}_location'] if 'barrier' in str(condition) else [])
    if len(homies) == 0:
        print(f"No homings of type {h_type} found for condition {condition} in dataset {dataset}. Skipping plot.")
        return
    # user choice: the width of the right heatmap is 200 frames to 25 bins in the left heatmap
    fig = plt.figure(figsize=(3*len(homies), 10))
    gs = gridspec.GridSpec(nrows=3, ncols=1, height_ratios=[1,2,2], hspace=0.15, wspace = 0.05)

    ax_neural = fig.add_subplot(gs[1])
    escape = plot_neural_data_homing(ax_neural, dataset, homies, all_data, tuning, settings)

    onset_c = results[f'{dataset}_frame_onsets'][homies]
    offsets_c = results[f'{dataset}_frame_offsets'][homies]
    durations_c = offsets_c - onset_c
    gs_top = gridspec.GridSpecFromSubplotSpec(1, len(onset_c), subplot_spec=gs[0], 
                                            width_ratios=durations_c, wspace=0.008)
    for i, h_idx in enumerate(homies):
        ax_xy = fig.add_subplot(gs_top[0, i])
        title = f"{dataset} homing {h_idx}"
        if escape[i]:
            title += ": Escape!"
        plot_mouse_behaviour(ax_xy,
                        x=vdf['mouse_x_position'].to_numpy(),
                        y=vdf['mouse_y_position'].to_numpy(),
                        onset=results[f'{dataset}_frame_onsets'][h_idx],
                        offset=results[f'{dataset}_frame_offsets'][h_idx],
                        condition=settings[f'replay_{dataset}_condition'],
                        barrier_coordinates=settings[f'barrier_{dataset}_location'] if 'barrier' in str(settings[f'replay_{dataset}_condition']) else None,
                        time_markers=[],
                        look_back=200,
                        look_forward=400,
                        title=title)

    ax_posterior = fig.add_subplot(gs[2])
    n_bins = results[f'{dataset}_{posterior}_posterior'].shape[1]
    posterior_dist = np.empty((n_bins, 0))
    predicted = np.array([])
    actual = np.array([])
    time_slice = np.array([])
    time_marker = [0]
    for h_idx in homies:
        posterior_dist = np.concatenate((posterior_dist, results[f'{dataset}_{posterior}_posterior'][results[f'{dataset}_position_onsets'][h_idx]:results[f'{dataset}_position_offsets'][h_idx], :].T), axis = 1)
        predicted = np.concatenate((predicted, results[f'{dataset}_predicted'][results[f'{dataset}_position_onsets'][h_idx]:results[f'{dataset}_position_offsets'][h_idx]]))
        actual = np.concatenate((actual, (data[f'{dataset}_position'][results[f'{dataset}_position_onsets'][h_idx]:results[f'{dataset}_position_offsets'][h_idx]]).ravel()))
        time_marker.append(((results[f'{dataset}_position_offsets'][h_idx] - results[f'{dataset}_position_onsets'][h_idx])/sampling_frequency) + time_marker[-1])
    time_slice = np.arange(posterior_dist.shape[1]) / sampling_frequency
    plot_predicted_vs_actual(ax_posterior, posterior_dist=posterior_dist, 
                            predicted_position=predicted, 
                            actual_position=actual,
                            time = time_slice, var_name = settings['replay_decoder_variable'], time_markers = time_marker)
    ax_posterior.set_xlim(0, time_slice[-1])

    fig.savefig(os.path.join(LM_path, f"{session_name}_{condition}_{dataset}_homings_{h_type}_across_conditions.pdf"), dpi=300)
    plt.close(fig)

In [ ]:
"""Run decoder!!"""
# combos = [[conditions[0], conditions[0]], [conditions[1], conditions[1]], [conditions[2], conditions[2]]] # same condition, different homing sets
combos = [[conditions[0], conditions[1]], [conditions[1], conditions[2]]] # diff condition, different homing sets

good_stuff = {}
for combo in combos:
    true_settings['replay_train_condition'] = combo[0]
    true_settings['replay_test_condition'] = combo[1]
    good_stuff[tuple(combo)] = {}
    for e in range(len(csv_paths)):
        # check a results csv, look for a settings match
        _, do_analysis, hexaname = check_database_for_same_run(
                                            db_settings=true_settings,
                                            results_csv_name=csv_paths[e],  # just check the first csv for now, we can loop through them later
                                            settings=Settings_ae,
                                        )
        if not do_analysis:
            folder_path = os.path.join("\\").join(csv_paths[e].split("\\")[:-1])
            settings_file = os.path.join(folder_path, "SSdecoder_" + hexaname + "_settings.npz")
            data_file = os.path.join(folder_path, "SSdecoder_" + hexaname + "_data.npz")
            save_file = os.path.join(folder_path, "SSdecoder_" + hexaname + "_results.npz")

            data = np.load(data_file, allow_pickle=True)
            settings = np.load(settings_file, allow_pickle=True)
            if data['train_spikes'].shape[0] == 0:
                print(f"Insufficient training data! Skipping session {session_names[e]}...")
            else:
                if len(np.where(np.diff(data['train_mask'].astype(int))>0)[0]) < 5:
                    print(f"Insufficient training data! Skipping session {session_names[e]}...")
                else:
                    print("Data loaded successfully for session:", session_names[e])
                    good_stuff[tuple(combo)][session_names[e]] = len(np.where(np.diff(data['train_mask'].astype(int))>0)[0])
                    # try:
                    #     results = np.load(save_file, allow_pickle=True)
                    # except FileNotFoundError:
                    run_decoder(data, settings, save_file)

for c in good_stuff.keys():
    print(c)
    for v, k in good_stuff[c].items():
        print(f" {v} {k}")

2026-03-16 14:57:42.536 | INFO     | behave_analysis.analyze.results_database_utils:check_database_for_same_run:23 - Found 1 matched results in database: ['6d3db33f34294181'] in the folder Z:\Jasmine_Laurence\Experimental_Data\JAL004\004_flip_puff2_2023_09_11T09_32_25\processed_data\models\replay\replay_SS_decoder\replay_results.csv


Data loaded successfully for session: JAL4_11thSept
Training time: 0.04 minutes


100%|██████████| 331/331 [00:00<00:00, 2184.59it/s]


Test prediction time: 2.23 minutes


100%|██████████| 331/331 [00:00<00:00, 975.28it/s]


...

Insufficient training data! Skipping session JAL8_14may...
Data loaded successfully for session: JAL8_flip4_10may
Training time: 0.00 minutes


100%|██████████| 65/65 [00:00<00:00, 1357.84it/s]


Test prediction time: 1.77 minutes


100%|██████████| 65/65 [00:00<00:00, 2043.47it/s]


Train prediction time: 0.31 minutes
('shelter_only', 'barrier_pre_flip')
 JAL4_11thSept 9
 JAL5_21stSept 13
 JAL6_28mar 6
 JAL6_flip4_21mar 15
 JAL6_flip3_18mar 31
 JAL6_flip5_25mar 17
 JAL7_flip5_22mar 11
 JAL7_flip2_12mar 11
 JAL7_sesh9_16apr 5
 JAL8_flip1_25apr 6
 JAL8_flip2_29apr 6
 JAL8_flip3_7may 14
 JAL8_14may 14
 JAL8_flip4_10may 9
('barrier_pre_flip', 'barrier_post_flip')
 JAL5_8thSept 8
 JAL6_28mar 9
 JAL6_flip4_21mar 28
 JAL6_flip3_18mar 10
 JAL6_flip5_25mar 9
 JAL7_sesh8_9apr 5
 JAL7_flip5_22mar 5
 JAL8_flip3_7may 14
 JAL8_flip4_10may 7


Let's make some plots for a chosen session

In [377]:
"""Load data for chosen session and condition to plot"""
combos = [[conditions[0], conditions[0]], [conditions[1], conditions[1]], [conditions[2], conditions[2]]] # same condition, different homing sets
combos = [[conditions[0], conditions[1]], [conditions[1], conditions[2]]] # diff condition, different homing sets

for e in np.arange(0,len(csv_paths)):
    session_name = session_names[e] # list(good_stuff[tuple(combo)].keys())[e]
    path_this_sesh = csv_paths[session_names.index(session_name)]
    video_loaded = False
    for combo in combos:
        true_settings['replay_train_condition'] = combo[0]
        true_settings['replay_test_condition'] = combo[1]
        _, do_analysis, hexaname = check_database_for_same_run(
                                                    db_settings=true_settings,
                                                    results_csv_name=path_this_sesh,  # just check the first csv for now, we can loop through them later
                                                    settings=Settings_ae,
                                                )
        if do_analysis:
            print("No matching run found in database. Please check your settings and try again.")
        else:
            folder_path = os.path.join("\\").join(path_this_sesh.split("\\")[:-1])
            settings_file = os.path.join(folder_path, "SSdecoder_" + hexaname + "_settings.npz")
            data_file = os.path.join(folder_path, "SSdecoder_" + hexaname + "_data.npz")
            save_file = os.path.join(folder_path, "SSdecoder_" + hexaname + "_results.npz")

            data = np.load(data_file, allow_pickle=True)
            settings = np.load(settings_file, allow_pickle=True)
            try:
                results = np.load(save_file, allow_pickle=True)
            except FileNotFoundError:
                print(f"Results file not found for session {session_name} and combo {combo}")
                continue
            sampling_frequency = 1 / settings['replay_state_space_decoder_bin_size'] # in Hz, bin_width is in s
            train_time = np.arange(0, data['train_time'].shape[0] / (sampling_frequency), settings['replay_state_space_decoder_bin_size'])
            test_time = np.arange(0, data['test_time'].shape[0] / (sampling_frequency), settings['replay_state_space_decoder_bin_size'])

            # load behavior data, filter with test mask, plot the trjectories and see what the mouse was actually doing!
            if not video_loaded:
                video_path = path_this_sesh.split("models")[0] + "full_video_dataframe.csv"
                vdf = pd.read_csv(video_path)
                video_loaded = True
            
            # load tuning data
            all_data = {}
            tuning = [true_settings['replay_decoder_variable'] + ' in ' + true_settings['replay_decoder_train_time_period'], 
                    true_settings['replay_decoder_variable'] + ' in ' + true_settings['replay_decoder_test_time_period']]
            tuning_path = os.path.join(path_this_sesh.split("models")[0], "models", "escape_tuning")
            exit_for_loop = False
            for tuna in tuning:
                _, do_analysis, hexaname = check_database_for_same_run(
                                                db_settings={"variable": tuna, **tuning_settings},
                                                results_csv_name=tuning_path + os.sep + "EscapePattern_results.csv",
                                                settings=Settings_ae,
                                            )
                if do_analysis:
                    print(f"Results for {tuna} not found in database, but should be! Check the database and settings.")
                    exit_for_loop = True
                    continue

                ep_data = np.load(os.path.join(tuning_path, "EPtuning_" + hexaname + "_results.npz"), allow_pickle=True)
                # 3. identify the significant cells in each condition
                real_stat, shift_stat = compute_tuning_stat(stat=stat, 
                                                            shifted_matrix=ep_data['fr_shift'], 
                                                            shift0=int(np.shape(ep_data['fr_shift'])[0] / 2), 
                                                            neural_matrix=ep_data['neural_matrix'], 
                                                            condition=ep_data['condition'])
                sig_escape = real_stat > np.nanpercentile(shift_stat, 95, axis=0)
                all_data[tuna] = {'real_stat': real_stat, 
                                    'shift_stat': shift_stat, 
                                    'sig_escape': sig_escape,
                                    'results': ep_data}
            if exit_for_loop:
                continue
            # plot the neural data for the homing bouts in the test set, sorted by tuning curve peak
            homie_overview_plot(types[0], 'train', vdf, results, all_data, tuning, settings, LM_path, true_settings['replay_train_condition'])
            if 'barrier' in str(settings['replay_test_condition']):
                for t in types[1:]:
                    homie_overview_plot(t, 'test', vdf, results, all_data, tuning, settings, LM_path, true_settings['replay_test_condition'])
            else:
                homie_overview_plot(types[0], 'test', vdf, results, all_data, tuning, settings, LM_path, true_settings['replay_test_condition'])
            

2026-03-17 20:41:26.844 | INFO     | behave_analysis.analyze.results_database_utils:check_database_for_same_run:23 - Found 1 matched results in database: ['6d3db33f34294181'] in the folder Z:\Jasmine_Laurence\Experimental_Data\JAL004\004_flip_puff2_2023_09_11T09_32_25\processed_data\models\replay\replay_SS_decoder\replay_results.csv


No matching run found in database. Please check your settings and try again.
No matching run found in database. Please check your settings and try again.
No matching run found in database. Please check your settings and try again.
No matching run found in database. Please check your settings and try again.
No matching run found in database. Please check your settings and try again.
No matching run found in database. Please check your settings and try again.


2026-03-17 20:42:54.428 | INFO     | behave_analysis.analyze.results_database_utils:check_database_for_same_run:23 - Found 1 matched results in database: ['ed45daad395e4f77'] in the folder Z:\Jasmine_Laurence\Experimental_Data\JAL004\004_flip_puff2_2023_09_11T09_32_25\processed_data\models\escape_tuning\EscapePattern_results.csv
2026-03-17 20:42:55.591 | INFO     | behave_analysis.analyze.results_database_utils:check_database_for_same_run:23 - Found 3 matched results in database: ['58fdd32f312141ba' '80cb632eb0c34dbe' '05bd5f2262a547b3'] in the folder Z:\Jasmine_Laurence\Experimental_Data\JAL004\004_flip_puff2_2023_09_11T09_32_25\processed_data\models\escape_tuning\EscapePattern_results.csv
2026-03-17 20:43:23.545 | INFO     | behave_analysis.analyze.results_database_utils:check_database_for_same_run:23 - Found 1 matched results in database: ['968870ba2aee4085'] in the folder Z:\Jasmine_Laurence\Experimental_Data\JAL004\004_flip_puff2_2023_09_11T09_32_25\processed_data\models\replay\re

Results file not found for session JAL4_11thSept and combo ['barrier_pre_flip', 'barrier_post_flip']
Results file not found for session JAL5_8thSept and combo ['shelter_only', 'barrier_pre_flip']


2026-03-17 20:44:41.088 | INFO     | behave_analysis.analyze.results_database_utils:check_database_for_same_run:23 - Found 1 matched results in database: ['7924958e054f4cb3'] in the folder Z:\Jasmine_Laurence\Experimental_Data\JAL005\005_flip1_2023_09_08T07_36_54\processed_data\models\escape_tuning\EscapePattern_results.csv
2026-03-17 20:44:42.171 | INFO     | behave_analysis.analyze.results_database_utils:check_database_for_same_run:23 - Found 3 matched results in database: ['f20c352bb7844aa2' 'b29c3569d1bc4dd7' '0c82f5bad498451f'] in the folder Z:\Jasmine_Laurence\Experimental_Data\JAL005\005_flip1_2023_09_08T07_36_54\processed_data\models\escape_tuning\EscapePattern_results.csv
2026-03-17 20:45:38.881 | INFO     | behave_analysis.analyze.results_database_utils:check_database_for_same_run:23 - Found 1 matched results in database: ['c7661bcd67774f52'] in the folder Z:\Jasmine_Laurence\Experimental_Data\JAL005\005_flippuff3_2023_09_21T11_11_13\processed_data\models\replay\replay_SS_dec

...

2026-03-17 21:19:31.830 | INFO     | behave_analysis.analyze.results_database_utils:check_database_for_same_run:23 - Found 1 matched results in database: ['8f70afefb8784ace'] in the folder Z:\Jasmine_Laurence\Experimental_Data\JAL008\JAL008_shelter_barrier_flip_2_2024_04_29T12_14_54\processed_data\models\escape_tuning\EscapePattern_results.csv
2026-03-17 21:19:32.851 | INFO     | behave_analysis.analyze.results_database_utils:check_database_for_same_run:23 - Found 2 matched results in database: ['c9f8e61a749441bb' '3fbf1a6a1cb0498f'] in the folder Z:\Jasmine_Laurence\Experimental_Data\JAL008\JAL008_shelter_barrier_flip_2_2024_04_29T12_14_54\processed_data\models\escape_tuning\EscapePattern_results.csv
2026-03-17 21:19:59.427 | INFO     | behave_analysis.analyze.results_database_utils:check_database_for_same_run:23 - Found 1 matched results in database: ['1f0aa8b65a4f4613'] in the folder Z:\Jasmine_Laurence\Experimental_Data\JAL008\JAL008_shelter_barrier_flip_2_2024_04_29T12_14_54\proce

Results file not found for session JAL8_flip2_29apr and combo ['barrier_pre_flip', 'barrier_post_flip']


2026-03-17 21:21:27.573 | INFO     | behave_analysis.analyze.results_database_utils:check_database_for_same_run:23 - Found 1 matched results in database: ['843c110ee21f441e'] in the folder Z:\Jasmine_Laurence\Experimental_Data\JAL008\JAL008_shelter_barrier_flip_3_2024_05_07T10_16_26\processed_data\models\escape_tuning\EscapePattern_results.csv
2026-03-17 21:21:28.611 | INFO     | behave_analysis.analyze.results_database_utils:check_database_for_same_run:23 - Found 2 matched results in database: ['7252e35101584c31' '4010dff9453d48b3'] in the folder Z:\Jasmine_Laurence\Experimental_Data\JAL008\JAL008_shelter_barrier_flip_3_2024_05_07T10_16_26\processed_data\models\escape_tuning\EscapePattern_results.csv
2026-03-17 21:22:26.093 | INFO     | behave_analysis.analyze.results_database_utils:check_database_for_same_run:23 - Found 1 matched results in database: ['6ea656f9ff6a4ded'] in the folder Z:\Jasmine_Laurence\Experimental_Data\JAL008\JAL008_shelter_barrier_flip_3_2024_05_07T10_16_26\proce

Results file not found for session JAL8_14may and combo ['barrier_pre_flip', 'barrier_post_flip']


2026-03-17 21:27:20.612 | INFO     | behave_analysis.analyze.results_database_utils:check_database_for_same_run:23 - Found 1 matched results in database: ['23b4a2470a2c477d'] in the folder Z:\Jasmine_Laurence\Experimental_Data\JAL008\JAL008_shelter_barrier_flip_4_2024_05_10T11_47_47\processed_data\models\escape_tuning\EscapePattern_results.csv
2026-03-17 21:27:21.033 | INFO     | behave_analysis.analyze.results_database_utils:check_database_for_same_run:23 - Found 2 matched results in database: ['7a545f0555704db4' '1b06c867da344f98'] in the folder Z:\Jasmine_Laurence\Experimental_Data\JAL008\JAL008_shelter_barrier_flip_4_2024_05_10T11_47_47\processed_data\models\escape_tuning\EscapePattern_results.csv
2026-03-17 21:27:54.267 | INFO     | behave_analysis.analyze.results_database_utils:check_database_for_same_run:23 - Found 1 matched results in database: ['283f9b66f6214898'] in the folder Z:\Jasmine_Laurence\Experimental_Data\JAL008\JAL008_shelter_barrier_flip_4_2024_05_10T11_47_47\proce